# Análise de Evasão Escolar na Grande Vitória (ES)

Este projeto tem como objetivo analisar os fatores associados à evasão escolar no Ensino Médio na região da Grande Vitória (Vitória, Cariacica, Vila Velha e Serra), utilizando dados do INEP.

A proposta é identificar padrões e relações entre desempenho acadêmico, tipo de rede de ensino e taxas de abandono escolar, buscando compreender quais variáveis estão mais associadas à evasão.

Nesta MVP serão apresentadas as seguintes etapas:

- Análise exploratória dos dados (EDA), incluindo comparações entre níveis de ensino, municípios e tipos de escola
- Análise estatística, incluindo correlação de Pearson para investigação de relações entre variáveis
- Modelagem supervisionada, com Regressão Linear Múltipla para interpretação dos fatores associados à evasão
- Modelagem preditiva com Random Forest Regressor para captura de relações não lineares
- Avaliação e comparação de desempenho dos modelos por meio de métricas como R², MAE e RMSE


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


Bibliotecas carregadas.


## Carregamento e Filtro Geográfico


In [ ]:
cols_micro = ['CO_ENTIDADE',' NO_MUNICIPIO', 'SG_UF', 'TP_DEPENDENCIA']
cidades = ['Vitória', 'Cariacica', 'Vila Velha', 'Serra']

try:
   df_micro = pd.read_csv('microdados_ed_basica_2024.csv', sep=';', encoding='latin1', usecols=cols_micro )
   df_es = df_micro[(df_micro['SG_UF'] == 'ES') & (df_micro['NO_MUNICIPIO'].isin(cidades))].copy()

   df_tx = pd.read_excel('tx_rend_escolas_2024.xlsx', skiprows=8)
   df_tx.columns = [c.strip() for c in df_tx.columns]

   df_tx['CO_ENTIDADE'] = df_tx['CO_ENTIDADE'].astype(str).str.split('.').str[0]
   df_es['CO_ENTIDADE'] = df_es['CO_ENTIDADE'].astype(str).str.split('.').str[0]

   df_final = pd.merge(df_es, df_tx, on='CO_ENTIDADE', how='inner')
        
   if df_final.empty:
           print("Atenção: O merge resultou em um dataframe vazio. Verifique os códigos das escolas.")
   else:
           print(f"Sucesso! {len(df_final)} escolas da Grande Vitória encontradas.")

except Exception as e:
    print(f"Erro ao processar: {e}")

Sucesso! 509 escolas da Grande Vitória encontradas.


## Conjunto de Dados Macro

In [ ]:
print(df_final.head())

In [ ]:
df_final.shape

In [ ]:
print(df_final.info())

## Verificação de dados nulos

In [ ]:
print(df_final.isnull().sum())

## Calculando as taxas de Abandono por Nível de Ensino

In [ ]:
cols_abandono_fund = [f'3_CAT_FUN_{i:02d}' for i in range(1, 10)]
cols_abandono_med = [f'3_CAT_MED_{i:02d}' for i in range(1, 4)]

df_final[cols_abandono_fund] = df_final[cols_abandono_fund].apply(
    pd.to_numeric,
    errors='coerce'
)

df_final[cols_abandono_med] = df_final[cols_abandono_med].apply(
    pd.to_numeric,
    errors='coerce'
)

In [ ]:
media_fund = df_final[cols_abandono_fund].stack().mean()
media_med = df_final[cols_abandono_med].stack().mean()

print(f"Média de abandono do Fundamental: {media_fund:.2f}")
print(f"Média de abandono do Médio: {media_med:.2f}")

## Correlação do Abandono com o Município

In [ ]:
cols_abandono = cols_abandono_fund + cols_abandono_med

df_final['taxa_abandono_total'] = df_final[cols_abandono].mean(axis=1)

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_final, 
            x='NO_MUNICIPIO_x', 
            y='taxa_abandono_total', 
            hue='NO_MUNICIPIO_x', 
            palette="Set2", 
            legend=False          
)

plt.title('Taxa de Abandono no Ensino Médio - Grande Vitória (2019)', fontsize=14)
plt.ylabel('Taxa de Abandono (%)')
plt.xlabel('Município')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

### Comparação da Taxa de Abandono: Fundamental x Médio

In [ ]:
df_final['taxa_abandono_fund'] = df_final[cols_abandono_fund].mean(axis=1)
df_final['taxa_abandono_med'] = df_final[cols_abandono_med].mean(axis=1)

comparativo_medias = df_final.groupby('NO_MUNICIPIO_x')[['taxa_abandono_fund', 'taxa_abandono_med']].mean()
print("Média de Abandono por Etapa e Município:")
display(comparativo_medias)

In [ ]:
df_melted = df_final.melt(id_vars=['NO_MUNICIPIO_x'], 
                         value_vars=['taxa_abandono_fund', 'taxa_abandono_med'],
                         var_name='Etapa', value_name='Taxa_Abandono')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='NO_MUNICIPIO_x', y='Taxa_Abandono', hue='Etapa', palette='muted')

plt.title('Comparação da Taxa de Abandono: Fundamental vs Médio', fontsize=14)
plt.ylabel('Taxa Média de Abandono (%)')
plt.xlabel('Município')
plt.legend(title='Etapa de Ensino')
plt.show()

No geral, a taxa média de abandono no ensino médio (1,13%) é aproximadamente 5,4 vezes maior que a observada no ensino fundamental (0,21%).

Isso pode estar associado a fatores como:
- entrada precoce no mercado de trabalho
- desmotivação com o modelo educacional
- aumento da dificuldade acadêmica

O Ensino Médio deve ser tratado como etapa crítica para políticas de permanência escolar.

## Fatores de Desempenho Escolar x Evasão

In [ ]:
cols_aprovacao_fund = [f'1_CAT_FUN_{i:02d}' for i in range(1, 10)]
cols_aprovacao_med = [f'1_CAT_MED_{i:02d}' for i in range(1, 4)]

df_final[cols_aprovacao_fund] = df_final[cols_aprovacao_fund].apply(
    pd.to_numeric,
    errors='coerce'
)

df_final[cols_aprovacao_med] = df_final[cols_aprovacao_med].apply(
    pd.to_numeric,
    errors='coerce'
)

cols_aprovacao = cols_aprovacao_fund + cols_aprovacao_med

df_final['taxa_aprovacao_total'] = df_final[cols_aprovacao].mean(axis=1)

### Correlação de Pearson entre a Taxa de Aprovação e a Taxa de Abandono

In [ ]:
df_final[['taxa_aprovacao_total', 'taxa_abandono_total']].corr()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_final, x='taxa_aprovacao_total', y='taxa_abandono_total', hue='NO_MUNICIPIO_x', s=100, alpha=0.7)
plt.title('Correlação: Taxa de Aprovação vs Taxa de Abandono')
plt.xlabel('Taxa de Aprovação (%)')
plt.ylabel('Taxa de Abandono (%)')
plt.axhline(df_final['taxa_abandono_total'].mean(), color='red', linestyle='--', label='Média de Abandono')
plt.legend()
plt.show()

Existe uma correlação negativa moderada entre aprovação e evasão. Em geral, municípios com maiores taxas de aprovação tendem a apresentar menores taxas de abandono escolar,  indicando que dificuldades de aprendizagem podem desestimular os alunos, consequentemente, à maior probabilidade de evasão escolar.

## Proporção de Abandono por tipo de escola 

In [ ]:
dict_dep = {1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'}
df_final['Rede'] = df_final['TP_DEPENDENCIA'].map(dict_dep)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_final, x='Rede', y='taxa_abandono_total', hue='Rede', palette='coolwarm', legend=False)
plt.title('Taxa de Abandono no Ensino Médio por Rede de Ensino')
plt.ylabel('Taxa de Abandono (%)')
plt.show()

Escolas públicas apresentam maior variabilidade e, em geral, maiores taxas de abandono em comparação às escolas privadas.

Isso pode refletir desigualdades socioeconômicas, onde alunos da rede pública enfrentam maiores desafios, como:
- necessidade de trabalhar
- menor apoio familiar
- menor acesso a recursos educacionais